In [0]:
%run /Workspace/Users/panchalraahul10@gmail.com/common/01_lakehouse_config

External lakehouse databases registered pointing to customer ADLS Gen2.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Generic Parameterized Transformation
# MAGIC Cleanses Bronze data and applies Delta Lake ACID Merges into `silver.cln_<entity>`.


# COMMAND ----------
# 2. Declare parameters via Databricks Widgets
dbutils.widgets.text("p_entity_name", "plans", "Entity Name")
dbutils.widgets.text("p_run_mode", "incremental", "Run Mode (full/incremental)")

entity_name = dbutils.widgets.get("p_entity_name").strip().lower()
run_mode = dbutils.widgets.get("p_run_mode").strip().lower()

print(f"Executing Silver transformation for entity: '{entity_name}' in mode: '{run_mode}'")

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# COMMAND ----------
# 3. Dynamic path resolution based on entity metadata
entity_paths = {
    "plans": {"bronze": f"{BRONZE_BASE}/sql_plans", "silver": f"{SILVER_BASE}/cln_plans", "table": "silver.cln_plans", "pk": ["plan_id"]},
    "organizations": {"bronze": f"{BRONZE_BASE}/sql_organizations", "silver": f"{SILVER_BASE}/cln_organizations", "table": "silver.cln_organizations", "pk": ["org_id"]},
    "users": {"bronze": f"{BRONZE_BASE}/sql_users", "silver": f"{SILVER_BASE}/cln_users", "table": "silver.cln_users", "pk": ["user_id"]},
    "subscriptions": {"bronze": f"{BRONZE_BASE}/sql_subscriptions", "silver": f"{SILVER_BASE}/cln_subscriptions", "table": "silver.cln_subscriptions", "pk": ["subscription_id"]},
    "invoices": {"bronze": f"{BRONZE_BASE}/sql_invoices", "silver": f"{SILVER_BASE}/cln_invoices", "table": "silver.cln_invoices", "pk": ["invoice_id"]},
    "feature_usage": {"bronze": f"{BRONZE_BASE}/file_feature_usage", "silver": f"{SILVER_BASE}/cln_feature_usage", "table": "silver.cln_feature_usage", "pk": ["user_id", "org_id", "usage_date", "feature_name"]},
    "logins": {"bronze": f"{BRONZE_BASE}/api_logins", "silver": f"{SILVER_BASE}/cln_logins", "table": "silver.cln_logins", "pk": ["event_id"]}
}

if entity_name not in entity_paths:
    raise ValueError(f"Entity '{entity_name}' is not recognized in silver transformation mapping.")

cfg = entity_paths[entity_name]

# Read raw Bronze parquet files across all ingestion subfolders
# (logins entity uses JSON — its branch reads data separately)
if entity_name != "logins":
    df_raw = spark.read.parquet(f"{cfg['bronze']}/*/*/*.parquet")
    print(f"Bronze records read: {df_raw.count()}")

# COMMAND ----------
# 4. Entity-specific transformations (replicates clean_sources.py logic)

if entity_name == "plans":
    df_clean = df_raw \
        .withColumn("plan_id", F.col("plan_id").cast("int")) \
        .withColumn("plan_name", F.trim(F.col("plan_name"))) \
        .withColumn("tier", F.lower(F.trim(F.col("tier")))) \
        .withColumn("monthly_fee", F.coalesce(F.col("monthly_fee").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("plan_id", "plan_name", "tier", "monthly_fee", "created_at", "updated_at")

elif entity_name == "organizations":
    # ISO-2 Country Standardizer Map
    country_map = F.create_map([
        F.lit("USA"), F.lit("US"), F.lit("United States"), F.lit("US"), F.lit("US"), F.lit("US"),
        F.lit("CAN"), F.lit("CA"), F.lit("Canada"), F.lit("CA"), F.lit("CA"), F.lit("CA"),
        F.lit("GBR"), F.lit("GB"), F.lit("United Kingdom"), F.lit("GB"), F.lit("GB"), F.lit("GB"),
        F.lit("DEU"), F.lit("DE"), F.lit("Germany"), F.lit("DE"), F.lit("DE"), F.lit("DE"),
        F.lit("IND"), F.lit("IN"), F.lit("India"), F.lit("IN"), F.lit("IN"), F.lit("IN")
    ])

    df_clean = df_raw \
        .withColumn("org_id", F.col("org_id").cast("int")) \
        .withColumn("name", F.trim(F.col("name"))) \
        .withColumn("industry", F.coalesce(F.when(F.trim(F.col("industry")) == "", "Unknown").otherwise(F.trim(F.col("industry"))), F.lit("Unknown"))) \
        .withColumn("country_code", F.coalesce(country_map[F.col("country")], F.lit("XX"))) \
        .withColumn("is_active", F.col("is_active").cast("boolean")) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("org_id", "name", "industry", "country_code", "is_active", "created_at", "updated_at")

elif entity_name == "users":
    # Email syntax validation pattern
    email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"

    df_clean = df_raw \
        .withColumn("user_id", F.col("user_id").cast("int")) \
        .withColumn("org_id", F.coalesce(F.col("org_id").cast("int"), F.lit(-1))) \
        .withColumn("cleaned_email", F.lower(F.trim(F.col("email")))) \
        .withColumn("is_valid_email", F.col("cleaned_email").rlike(email_regex)) \
        .withColumn("email", F.when(F.col("is_valid_email"), F.col("cleaned_email")).otherwise("invalid_user@system.local")) \
        .withColumn("role", F.coalesce(F.trim(F.col("role")), F.lit("Member"))) \
        .withColumn("is_active", F.col("is_active").cast("boolean")) \
        .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("user_id", "org_id", "email", "is_valid_email", "role", "is_active", "created_at", "updated_at")

elif entity_name == "subscriptions":
    df_clean = df_raw \
        .withColumn("subscription_id", F.col("subscription_id").cast("int")) \
        .withColumn("org_id", F.col("org_id").cast("int")) \
        .withColumn("plan_id", F.col("plan_id").cast("int")) \
        .withColumn("status", F.lower(F.trim(F.col("status")))) \
        .withColumn("started_at", F.to_timestamp(F.col("started_at"))) \
        .withColumn("cancelled_at", F.to_timestamp(F.col("cancelled_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .select("subscription_id", "org_id", "plan_id", "status", "started_at", "cancelled_at", "updated_at")

elif entity_name == "invoices":
    df_clean = df_raw \
        .withColumn("invoice_id", F.col("invoice_id").cast("int")) \
        .withColumn("subscription_id", F.col("subscription_id").cast("int")) \
        .withColumn("amount_due", F.coalesce(F.col("amount_due").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("amount_paid", F.coalesce(F.col("amount_paid").cast("decimal(10,2)"), F.lit(0.00))) \
        .withColumn("status", F.lower(F.trim(F.col("status")))) \
        .withColumn("billing_date", F.to_date(F.col("billing_date"))) \
        .withColumn("paid_at", F.to_timestamp(F.col("paid_at"))) \
        .withColumn("updated_at", F.to_timestamp(F.col("updated_at"))) \
        .withColumn("is_anomaly", F.col("amount_paid") > F.col("amount_due")) \
        .select("invoice_id", "subscription_id", "amount_due", "amount_paid", "status", "billing_date", "paid_at", "is_anomaly", "updated_at")

elif entity_name == "feature_usage":
    # Parse mixed date formats without crashing (YYYY-MM-DD vs MM/DD/YYYY vs DD-MM-YYYY)
    raw_date_col = F.col("UsageDate")
    parsed_date = F.coalesce(
        F.expr("try_to_date(UsageDate, 'yyyy-MM-dd')"),
        F.expr("try_to_date(UsageDate, 'MM/dd/yyyy')"),
        F.expr("try_to_date(UsageDate, 'dd-MM-yyyy')")
    )

    # Strip currency signs, commas, and negative values
    cleaned_usage_count = F.regexp_replace(F.col("UsageCount").cast("string"), r"[\$,]", "")

    df_clean = df_raw \
        .withColumnRenamed("UserId", "user_id") \
        .withColumnRenamed("OrgId", "org_id") \
        .withColumnRenamed("FeatureName", "feature_name") \
        .withColumn("user_id", F.coalesce(F.col("user_id").cast("int"), F.lit(-1))) \
        .withColumn("org_id", F.coalesce(F.col("org_id").cast("int"), F.lit(-1))) \
        .withColumn("usage_date", parsed_date) \
        .withColumn("feature_name", F.coalesce(F.when(F.trim(F.col("feature_name")) == "", "unknown_feature").otherwise(F.trim(F.col("feature_name"))), F.lit("unknown_feature"))) \
        .withColumn("usage_count", F.coalesce(cleaned_usage_count.cast("int"), F.lit(0))) \
        .filter(F.col("usage_count") >= 0) \
        .dropDuplicates(["user_id", "org_id", "usage_date", "feature_name"]) \
        .select("user_id", "org_id", "usage_date", "feature_name", "usage_count")

elif entity_name == "logins":
    # 1. Read the raw JSON file directly from Bronze
    # Spark automatically infers: root -> source, batch_id, record_count, data (array of structs)
    df_raw_json = spark.read.option("multiline", "true").json(f"{cfg['bronze']}/*/*/*.json")
    
    # 2. Explode the nested 'data' array so every event becomes a distinct row
    df_exploded = df_raw_json.select(
        F.explode("data").alias("login_event"),
        F.col("_ingested_at").alias("_batch_ingested_at") if "_ingested_at" in df_raw_json.columns else F.current_timestamp().alias("_batch_ingested_at")
    ).select(
        F.col("login_event.event_id").alias("event_id"),
        F.col("login_event.user_uid").alias("user_uid"),
        F.col("login_event.ip_address").alias("ip_address"),
        F.col("login_event.client_device").alias("client_device"),
        F.col("login_event.login_success").alias("login_success"),
        F.col("login_event.event_timestamp").alias("event_timestamp")
    )
    
    print(f"Raw exploded login events: {df_exploded.count()}")  # Will display 10,300!
    
    # 3. Apply the Data Quality & Cleaning Rules (Deduplication, UTC parsing, Null handling)
    # Deduplicate network retries retaining newest event_timestamp
    window_spec = Window.partitionBy("event_id").orderBy(F.col("event_timestamp").desc())
    
    df_clean = df_exploded \
        .withColumnRenamed("user_uid", "user_id") \
        .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .drop("row_num") \
        .withColumn("user_id", F.coalesce(F.col("user_id").cast("int"), F.lit(-1))) \
        .withColumn("ip_address", F.trim(F.col("ip_address"))) \
        .withColumn("client_device", F.coalesce(F.trim(F.col("client_device")), F.lit("Unknown"))) \
        .withColumn("login_success", F.col("login_success").cast("boolean")) \
        .withColumn("event_timestamp", F.to_utc_timestamp(F.to_timestamp(F.col("event_timestamp")), "UTC")) \
        .filter(F.col("event_timestamp").isNotNull()) \
        .select("event_id", "user_id", "ip_address", "client_device", "login_success", "event_timestamp")

# COMMAND ----------
# 5. External Delta File Upsert via Path Only (Zero Metastore Dependencies)

target_path = cfg["silver"]
pk_cols = cfg["pk"]
match_predicate = " AND ".join([f"target.{col} = source.{col}" for col in pk_cols])

if not DeltaTable.isDeltaTable(spark, target_path):
    print(f"Creating initial Delta files at {target_path}...")
    df_clean.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)
    print(f"Delta files initialized at {target_path}.")
else:
    print(f"Merging updates into {target_path}...")
    delta_target = DeltaTable.forPath(spark, target_path)
    
    delta_target.alias("target").merge(
        df_clean.alias("source"),
        match_predicate
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge completed successfully.")

# COMMAND ----------
# 6. Audit & Time Travel Verification
display(spark.sql(f"DESCRIBE HISTORY delta.`{target_path}` LIMIT 5"))

Executing Silver transformation for entity: 'feature_usage' in mode: 'incremental'
Bronze records read: 12480
Creating initial Delta files at abfss://silver@datalakesaas.dfs.core.windows.net/cln_feature_usage...
Delta files initialized at abfss://silver@datalakesaas.dfs.core.windows.net/cln_feature_usage.


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-13T07:18:07Z,146452211828492,panchalraahul10@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(4137456372015682),1012-081722-b4cu5evv,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 11629, numOutputBytes -> 77246)",null,Databricks-Runtime/17.3.x-scala2.13


In [0]:
# # Your list of table names
# arr = ['cln_feature_usage', 'cln_logins', 'cln_users', 'cln_organizations', 'cln_invoices', 'cln_plans', 'cln_subscriptions']

# # Loop through each table and execute the count
# for table in arr:
#     query = f"SELECT count(*) AS total_{table} FROM delta.`abfss://silver@datalakesaas.dfs.core.windows.net/{table}`"
    
#     print(f"Running query for {table}...")
#     df = spark.sql(query)
    
#     # Display the result (use df.show() in standard Spark or display(df) in Databricks)
#     df.show()


Running query for cln_feature_usage...
+-----------------------+
|total_cln_feature_usage|
+-----------------------+
|                  11629|
+-----------------------+

Running query for cln_logins...
+----------------+
|total_cln_logins|
+----------------+
|           10000|
+----------------+

Running query for cln_users...
+---------------+
|total_cln_users|
+---------------+
|           2500|
+---------------+

Running query for cln_organizations...
+-----------------------+
|total_cln_organizations|
+-----------------------+
|                    300|
+-----------------------+

Running query for cln_invoices...
+------------------+
|total_cln_invoices|
+------------------+
|               992|
+------------------+

Running query for cln_plans...
+---------------+
|total_cln_plans|
+---------------+
|              5|
+---------------+

Running query for cln_subscriptions...
+-----------------------+
|total_cln_subscriptions|
+-----------------------+
|                    344|
+----

In [0]:
# %sql
# SELECT * FROM delta.`abfss://silver@datalakesaas.dfs.core.windows.net/cln_feature_usage`

user_id,org_id,usage_date,feature_name,usage_count
2302,105,2026-07-03,api_calls,257
2424,85,2026-06-12,webhook_trigger,249
1574,145,2026-04-27,report_export,290
1319,126,2026-06-13,webhook_trigger,441
480,207,2026-08-28,report_export,385
370,148,2026-04-11,seat_added,108
1587,235,2026-09-05,seat_added,500
2106,195,2026-08-22,seat_added,330
66,156,2026-09-17,seat_added,222
223,185,2026-05-21,dashboard_view,263
